# Crash Course — Genomics (유전체)

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/genomics.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/genomics.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 예시 1 — 주석 달린 변이 표를 단계적으로 좁히기

In [ ]:
import pandas as pd

# 가상 변이 표: 주석까지 끝난 VCF를 표로 펼친 모습 (값은 모두 지어낸 것)
variants = pd.DataFrame({
    "chrom":  ["chr1", "chr1", "chr2", "chr7", "chr12", "chr17", "chr19", "chrX"],
    "pos":    [155235, 155240, 47403, 117559, 40340, 43093, 44908, 15372],
    "ref":    ["G", "C", "A", "CTT", "G", "T", "T", "A"],
    "alt":    ["A", "T", "G", "C", "A", "TA", "C", "G"],
    "gene":   ["GENE_A", "GENE_A", "GENE_B", "GENE_C", "GENE_D", "GENE_E", "GENE_F", "GENE_G"],
    "consequence": ["missense", "synonymous", "missense", "frameshift",
                    "missense", "frameshift", "missense", "stop_gained"],
    "QUAL":   [812, 45, 630, 990, 18, 240, 1500, 77],   # 변이 품질 점수
    "DP":     [42, 35, 8, 51, 30, 12, 60, 6],           # 그 자리를 덮은 read 수
    "FILTER": ["PASS", "PASS", "PASS", "PASS", "LowQual", "PASS", "PASS", "PASS"],
    "pop_AF": [0.00002, 0.21, 0.0004, None, 0.0001, 0.00001, 0.14, 0.0],  # 집단 빈도, None = DB에 없음
})

print("전체 변이 수:", len(variants))

# 1단계 — 기술적 필터: 호출 자체를 믿을 수 있는가
tech = variants[(variants["FILTER"] == "PASS") & (variants["QUAL"] >= 30) & (variants["DP"] >= 10)]
print("기술적 필터 통과:", len(tech))

# 2단계 — 빈도 필터: 희귀 질환 후보라면 흔한 변이는 뺀다.
# DB에 없는 변이(None)는 0으로 간주해 '남긴다' — 지우면 가장 희귀한 변이를 잃는다.
rare = tech[tech["pop_AF"].fillna(0) < 0.001]
print("희귀(AF < 0.001) 변이:", len(rare))

# 3단계 — 기능 필터: 단백질을 바꾸는 변이만
damaging = ["missense", "frameshift", "stop_gained", "splice_site"]
cand = rare[rare["consequence"].isin(damaging)]
print("단백질 변화 변이:", len(cand))
print()
print(cand[["chrom", "pos", "ref", "alt", "gene", "consequence", "DP", "pop_AF"]].to_string(index=False))

# 자기 점검: 단계가 진행될수록 수가 줄어야 하고, 빈도 미상 변이는 남아 있어야 한다
assert len(variants) >= len(tech) >= len(rare) >= len(cand)
assert cand["pop_AF"].isna().sum() == 1

### 예시 2 — 유전자형에서 대립유전자 빈도와 case/control 연관 검정

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(7)

# 가상 SNP 하나: 유전자형을 '대체 대립유전자(alt allele) 개수' 0/1/2로 적는다
# 환자군의 alt 빈도를 0.30, 대조군을 0.20으로 두고 뽑는다 (지어낸 값)
n_case, n_ctrl = 500, 500
case = rng.binomial(2, 0.30, n_case)
ctrl = rng.binomial(2, 0.20, n_ctrl)

geno = pd.DataFrame({
    "group": ["case"] * n_case + ["control"] * n_ctrl,
    "alt_count": np.concatenate([case, ctrl]),
})

# 유전자형 개수 표
print(pd.crosstab(geno["group"], geno["alt_count"]))
print()

# 대립유전자 빈도 = alt 대립유전자 수 / 전체 대립유전자 수(사람 수 x 2)
af = geno.groupby("group")["alt_count"].sum() / (geno.groupby("group")["alt_count"].count() * 2)
print("alt 대립유전자 빈도")
print(af.round(3).to_string())
print()

# 대립유전자 단위 2x2 표: 행 = 군, 열 = (alt, ref)
alt = geno.groupby("group")["alt_count"].sum()
tot = geno.groupby("group")["alt_count"].count() * 2
table = np.array([[alt["case"], tot["case"] - alt["case"]],
                  [alt["control"], tot["control"] - alt["control"]]])
chi2, p, dof, _ = stats.chi2_contingency(table, correction=False)
odds_ratio = (table[0, 0] * table[1, 1]) / (table[0, 1] * table[1, 0])
print(f"chi2 = {chi2:.2f}, p = {p:.2e}, odds ratio = {odds_ratio:.2f}")
print("유전체 수준 유의성(5e-8) 통과:", p < 5e-8)

# QC 한 가지: 대조군의 Hardy-Weinberg 평형 (관찰 유전자형 수 vs 기대 수)
q = af["control"]
obs = np.bincount(ctrl, minlength=3)
exp = np.array([(1 - q) ** 2, 2 * q * (1 - q), q ** 2]) * n_ctrl
hwe_chi2 = ((obs - exp) ** 2 / exp).sum()
print(f"대조군 HWE: chi2 = {hwe_chi2:.2f}, p = {stats.chi2.sf(hwe_chi2, df=1):.2f}")

assert table.sum() == 2 * (n_case + n_ctrl)